# Project 2: pretrained embeddings on ISEAR
Word embeddings with two models (embed-and-pool and LSTM), compared with our best project 1 model (57.7%).

| Section | Who |
| --- | --- |
| 0. Data and folds | Britt |
| 1. Embedding model, 2. What the embeddings capture | Britt |
| 3. Embed-and-pool | |
| 4. LSTM | |
| 5. Hyperparameter search and variance, 6. Error analysis | Together |

Run `python explore_data.py` first.

In [14]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedKFold

import part1
import part2

results_folder = Path("results/project2")
results_folder.mkdir(parents=True, exist_ok=True)

## 0. Data and folds
Same data and folds as project 1, so we can compare directly with last week's results.

In [15]:
LABELS = ["joy", "fear", "anger", "sadness", "disgust", "shame", "guilt"]
outcome_space = part1.clslib.NamedClasses(LABELS)

df = pd.read_csv("data/isear_clean.csv")
pairs = list(zip(df["text"], df["label_id"]))  # (text, label) pairs, as in the tutorials
print(len(pairs), "texts")

def stratified_cv_split(pairs, k, seed):
    """Same split as project 1: every fold has the same mix of emotions."""
    labels = [y for _, y in pairs]
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    return {fold: {"training": [pairs[i] for i in tr], "validation": [pairs[i] for i in va]}
            for fold, (tr, va) in enumerate(skf.split(np.zeros(len(pairs)), labels))}

seeds = [0, 3, 5]
num_folds = 5
cv_sets = [stratified_cv_split(pairs, k=num_folds, seed=seed) for seed in seeds]

7411 texts


In [16]:
# Check: are these exactly the project 1 folds? (compares the validation texts of every seed and fold)
p1 = pd.read_csv("results/project1/predictions.csv")
p1 = p1[(p1["representation"] == "bow+bigrams") & (p1["architecture"] == "loglinear")]
row_of = {text: i for i, text in enumerate(df["text"])}

for seed, cv in zip(seeds, cv_sets):
    for fold, split in cv.items():
        ours = sorted(row_of[text] for text, _ in split["validation"])
        theirs = sorted(p1[(p1["seed"] == seed) & (p1["fold"] == fold)]["row"])
        assert ours == theirs, f"seed {seed} fold {fold} differs from project 1"
print("Folds are identical to project 1")

Folds are identical to project 1


## 1. Embedding model (Britt)
We compare GloVe (Wikipedia + Gigaword, 50 dimensions; the 2014 version from Stanford's own Hugging Face page, because Stanford's server for the 2024 version was unreachable) and fastText (300 dimensions). The first run downloads them into `embeddings/` (a few minutes). If a download fails, download the file in your browser and put it in `embeddings/` (the error message says which file).

In [17]:
from collections import Counter
import shutil
from urllib.request import Request, urlopen
from zipfile import ZipFile

import nltk
from sklearn.linear_model import LogisticRegression

nltk.download("punkt_tab", quiet=True)  # used by the tokeniser
nltk.download("stopwords", quiet=True)

COMPARE_MODELS = True  # True: load both models (sections 1-2). False: only CHOSEN, which is enough for sections 3-4
CHOSEN = "fasttext"    # TODO: set to the model chosen from the comparison below

emb_folder = Path("embeddings")
emb_folder.mkdir(exist_ok=True)

# (download url, file to use inside the zip or None if it holds one file, dimensions, file starts with a header line)
EMB_OPTIONS = {
    "glove": ("https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip", "glove.6B.50d.txt", 50, False),
    "glove300": ("https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip", "glove.6B.300d.txt", 300, False),
    "fasttext": ("https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip", None, 300, True),
}

def download(url, path):
    """Download url to path. On failure, remove the half-finished file and say how to do it by hand."""
    print(f"downloading {path.name} ...")
    try:
        with urlopen(Request(url, headers={"User-Agent": "Mozilla/5.0"}), timeout=60) as r, open(path, "wb") as f:
            shutil.copyfileobj(r, f)
    except Exception as e:
        path.unlink(missing_ok=True)  # a half-finished zip would otherwise look like a finished one
        raise RuntimeError(f"Could not download {url} ({e}).\n"
                           f"Download it in your browser and put it in {emb_folder}/ as {path.name}") from None

def load_embeddings(name):
    """Download once, then load with the tutorial's PretrainedEmbeddings (which lowercases every word)."""
    url, member, dims, skip_first = EMB_OPTIONS[name]
    zip_path = emb_folder / url.split("/")[-1]
    file_path = emb_folder / member if member else zip_path
    if not file_path.exists():
        if not zip_path.exists():
            download(url, zip_path)
        if member:  # the GloVe zip holds 4 sizes; take out the one we use
            with ZipFile(zip_path) as z:
                z.extract(member, emb_folder)
    return part2.clslib.PretrainedEmbeddings(str(file_path), dims, skip_first_line=skip_first)

emb_models = {name: load_embeddings(name) for name in (EMB_OPTIONS if COMPARE_MODELS else [CHOSEN])}
for name, emb in emb_models.items():
    print(f"{name}: {len(emb.vocab):,} words, {emb.vectors.shape[1]} dimensions")

glove: 400,002 words, 50 dimensions
glove300: 400,002 words, 300 dimensions
fasttext: 830,463 words, 300 dimensions


### Tokenisation
Word-level tokens, lowercased like the embeddings. For embed-and-pool we remove punctuation and stopwords; for the LSTM we keep everything.

In [18]:
texts = [text for text, _ in pairs]

class FixedTokenizer(part2.clslib.BasicTokenizer):
    """The tutorial's tokenizer, but NLTK's split-off "n't" (from "didn't") and its quote symbols become words the embeddings know."""
    FIX = {"n't": "not", "``": '"', "''": '"'}

    def tokens(self, text):
        return [self.FIX.get(t, t) for t in super().tokens(text)]

def make_tokenizers(emb):
    """Return (tokenizer for embed-and-pool, tokenizer for the LSTM), both matching the embedding vocabulary."""
    pool = FixedTokenizer(lowercase=True, remove_punct=True, remove_stopwords=True, vocab=emb.vocab)
    rnn = FixedTokenizer(lowercase=True, remove_punct=False, remove_stopwords=False, vocab=emb.vocab)
    return pool, rnn

print("negations in the stopword list:", [w for w in ["not", "no", "nor", "never", "didn't", "don't"]
                                          if w in set(nltk.corpus.stopwords.words("english"))])
example = df["text"][0]
print("\n" + example)
for tok_name, tok in zip(["pool", "rnn"], make_tokenizers(emb_models[CHOSEN])):
    print(f"{tok_name}: {tok.tokens(example)}")

negations in the stopword list: ['not', 'no', 'nor', "didn't", "don't"]

On days when I feel close to my partner and other friends. When I feel at peace with myself and also experience a close contact with people whom I regard greatly.
pool: ['days', 'feel', 'close', 'partner', 'friends', 'feel', 'peace', 'also', 'experience', 'close', 'contact', 'people', 'regard', 'greatly']
rnn: ['on', 'days', 'when', 'i', 'feel', 'close', 'to', 'my', 'partner', 'and', 'other', 'friends', '.', 'when', 'i', 'feel', 'at', 'peace', 'with', 'myself', 'and', 'also', 'experience', 'a', 'close', 'contact', 'with', 'people', 'whom', 'i', 'regard', 'greatly', '.']


### Coverage
How many of our words does each model know? Unknown words lose their meaning.

In [19]:
def coverage(emb, tokenizer):
    """Share of tokens and word types found in the embedding vocabulary, plus the missing words with counts."""
    tokenised = [tokenizer.tokens(t) for t in texts]
    counts = Counter(tok for toks in tokenised for tok in toks)
    missing = Counter({w: c for w, c in counts.items() if w not in emb})
    stats = {
        "tokens covered": 1 - sum(missing.values()) / sum(counts.values()),
        "word types covered": 1 - len(missing) / len(counts),
        "texts with an unknown word": np.mean([any(tok not in emb for tok in toks) for toks in tokenised]),
    }
    return stats, missing

rows, missing_words = [], {}
for name, emb in emb_models.items():
    for tok_name, tok in zip(["embed-and-pool", "lstm"], make_tokenizers(emb)):
        stats, missing = coverage(emb, tok)
        rows.append({"embeddings": name, "tokeniser": tok_name, **{k: round(v, 3) for k, v in stats.items()}})
        if tok_name == "lstm":
            missing_words[name] = missing
coverage_table = pd.DataFrame(rows)
coverage_table.to_csv(results_folder / "coverage.csv", index=False)
coverage_table

,embeddings,tokeniser,tokens covered,word types covered,texts with an unknown word
0,glove,embed-and-pool,0.990,0.925,0.095
1,glove,lstm,0.994,0.899,0.140
2,glove300,embed-and-pool,0.990,0.925,0.095
3,glove300,lstm,0.994,0.899,0.140
4,fasttext,embed-and-pool,0.993,0.954,0.062
5,fasttext,lstm,0.996,0.937,0.087


In [20]:
for name, missing in missing_words.items():
    print(f"{name}: {len(missing):,} missing word types; most frequent: {missing.most_common(20)}\n")

emotion_words = LABELS + ["happy", "afraid", "angry", "sad", "disgusted", "ashamed", "guilty"]
for name, emb in emb_models.items():
    print(f"{name}: emotion words missing from the vocabulary: {[w for w in emotion_words if w not in emb]}")

glove: 949 missing word types; most frequent: [('examn', 28), ('girl-friend', 18), ('room-mate', 15), ('boy-friend', 14), ('class-mate', 13), ('fianct', 12), ('quarelled', 7), ('class-mates', 7), ('vomitted', 6), ('bus-stop', 5), ('assulted', 5), ('beated', 4), ('n.z', 4), ('catched', 4), ('gottenburg', 3), ('shop-assistant', 3), ('rejoyced', 3), ('life-partner', 3), ('enveloppe', 3), ('collegue', 3)]

glove300: 949 missing word types; most frequent: [('examn', 28), ('girl-friend', 18), ('room-mate', 15), ('boy-friend', 14), ('class-mate', 13), ('fianct', 12), ('quarelled', 7), ('class-mates', 7), ('vomitted', 6), ('bus-stop', 5), ('assulted', 5), ('beated', 4), ('n.z', 4), ('catched', 4), ('gottenburg', 3), ('shop-assistant', 3), ('rejoyced', 3), ('life-partner', 3), ('enveloppe', 3), ('collegue', 3)]

fasttext: 589 missing word types; most frequent: [('examn', 28), ("o'clock", 17), ('fianct', 12), ('1/2', 8), ('quarelled', 7), ('gottenburg', 3), ('shop-assistant', 3), ('rejoyced', 3)

### Quick task check
Average the word vectors of each text and train logistic regression: which embeddings work better for our task?

In [21]:
y_all = np.array([y for _, y in pairs])

def mean_pooled(emb, tokenizer):
    """One vector per text: the average of its known word vectors."""
    X = np.zeros((len(texts), emb.vectors.shape[1]), dtype=np.float32)
    for i, t in enumerate(texts):
        ids = [j for j in tokenizer.ids(t) if j != emb.vocab.default_index]
        if ids:
            X[i] = emb.vectors[ids].mean(0)
    return X

for name, emb in emb_models.items():
    X = mean_pooled(emb, make_tokenizers(emb)[0])
    accs = []
    for split in cv_sets[0].values():
        tr = [row_of[t] for t, _ in split["training"]]
        va = [row_of[t] for t, _ in split["validation"]]
        accs.append(LogisticRegression(max_iter=2000).fit(X[tr], y_all[tr]).score(X[va], y_all[va]))
    print(f"{name}: accuracy {np.mean(accs):.3f} ± {np.std(accs):.3f}")

glove: accuracy 0.422 ± 0.009
glove300: accuracy 0.541 ± 0.002
fasttext: accuracy 0.562 ± 0.004


In [22]:
emb_model = emb_models[CHOSEN]
tokenizer_for_pool, tokenizer_for_rnn = make_tokenizers(emb_model)
print(f"using {CHOSEN}: {len(emb_model.vocab):,} words, {emb_model.vectors.shape[1]} dimensions")

using fasttext: 830,463 words, 300 dimensions


## **Conclusion Q1:**

We use **fastText**. It knows 94% of our word types (GloVe 90%), leaves fewer texts with an unknown word (8.7% vs 14.0%), and knows common misspellings like "affraid". In the quick check it beats GloVe at the same size (56.2% vs 54.1% at 300 dimensions), so the difference is not only size. We map NLTK's "n't" to "not" so the LSTM keeps negations.

## 2. What the embeddings capture (Britt)
Which words are close to the emotion words?

In [23]:
def cosine(emb, a, b):
    """Cosine similarity of two words; NaN if either is not in the vocabulary (it would get the unk vector)."""
    if a not in emb or b not in emb:
        return float("nan")
    u, v = emb.embed_word(a), emb.embed_word(b)
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-8))

def neighbours(emb, word, k=8):
    """The k words closest to `word` by cosine similarity (pad and unk excluded)."""
    norms = np.linalg.norm(emb.vectors, axis=1) + 1e-8
    v = emb.embed_word(word)
    sims = (emb.vectors @ v) / (norms * (np.linalg.norm(v) + 1e-8))
    sims[:2] = -np.inf                  # ids 0 and 1 are pad and unk
    sims[emb.index(word)] = -np.inf     # the word itself
    return [emb.word(int(i)) for i in np.argsort(-sims)[:k]]

for name, emb in emb_models.items():
    print(f"# {name}: nearest neighbours")
    for w in ["happy", "afraid", "angry", "sad", "disgusted", "ashamed", "guilty"]:
        print(f"{w:>10}: {', '.join(neighbours(emb, w)) if w in emb else '(not in vocabulary)'}")
    print()

# glove: nearest neighbours
     happy: 'm, everyone, everybody, really, me, definitely, maybe, 'd
    afraid: scared, anybody, nobody, tell, anymore, feel, know, anyone
     angry: furious, outraged, enraged, anger, angered, frustrated, shouted, loud
       sad: awful, sorry, terrible, sadly, feeling, horrible, feels, tragic
 disgusted: ashamed, embarrassed, appalled, annoyed, amused, disheartened, intimidated, agitated
   ashamed: disgusted, embarrassed, appalled, sorry, honestly, offended, frankly, mortified
    guilty: pleaded, convicted, charges, charged, acquitted, conviction, murder, sentenced

# glove300: nearest neighbours
     happy: 'm, glad, pleased, really, always, everyone, everybody, feel
    afraid: scared, fearful, worried, fear, frightened, want, tell, know
     angry: enraged, furious, irate, outraged, frustrated, angered, provoked, annoyed
       sad: sorry, tragic, happy, unfortunate, terribly, pathetic, awful, poignant
 disgusted: appalled, horrified, dismayed, di

### Similarity between emotion names
Are the emotions we confused in project 1 (shame/guilt, anger/disgust) also close here?

In [24]:
for name, emb in emb_models.items():
    table = pd.DataFrame([[cosine(emb, a, b) for b in LABELS] for a in LABELS], index=LABELS, columns=LABELS)
    print(f"# {name}: cosine similarity between emotion names")
    print(table.round(2).to_string(), "\n")

# glove: cosine similarity between emotion names
          joy  fear  anger  sadness  disgust  shame  guilt
joy      1.00  0.45   0.58     0.80     0.60   0.68   0.50
fear     0.45  1.00   0.85     0.56     0.59   0.72   0.63
anger    0.58  0.85   1.00     0.74     0.84   0.74   0.62
sadness  0.80  0.56   0.74     1.00     0.78   0.74   0.64
disgust  0.60  0.59   0.84     0.78     1.00   0.77   0.61
shame    0.68  0.72   0.74     0.74     0.77   1.00   0.72
guilt    0.50  0.63   0.62     0.64     0.61   0.72   1.00 

# glove300: cosine similarity between emotion names
          joy  fear  anger  sadness  disgust  shame  guilt
joy      1.00  0.34   0.38     0.61     0.35   0.44   0.28
fear     0.34  1.00   0.61     0.39     0.39   0.48   0.38
anger    0.38  0.61   1.00     0.61     0.68   0.53   0.43
sadness  0.61  0.39   0.61     1.00     0.59   0.60   0.46
disgust  0.35  0.39   0.68     0.59     1.00   0.52   0.34
shame    0.44  0.48   0.53     0.60     0.52   1.00   0.49
guilt    0.2

### Opposites vs. synonyms
Are opposites (happy/sad) as close as synonyms (happy/glad)? That would be a problem for our task.

In [25]:
word_pairs = [("happy", "glad"), ("happy", "sad"), ("angry", "furious"), ("angry", "calm"),
              ("love", "hate"), ("proud", "ashamed"), ("guilty", "innocent"), ("ashamed", "embarrassed")]
pd.DataFrame([{"pair": f"{a} / {b}", **{name: round(cosine(emb, a, b), 2) for name, emb in emb_models.items()}}
              for a, b in word_pairs])

,pair,glove,glove300,fasttext
0,happy / glad,0.87,0.69,0.75
1,happy / sad,0.69,0.57,0.70
2,angry / furious,0.87,0.71,0.75
3,angry / calm,0.63,0.35,0.51
4,love / hate,0.65,0.45,0.71
5,proud / ashamed,0.68,0.57,0.72
6,guilty / innocent,0.73,0.58,0.70
7,ashamed / embarrassed,0.86,0.73,0.82


## **Conclusion Q2:** 

FastText's neighbours are real emotion synonyms, while GloVe learned the legal sense of "guilty" (pleaded, convicted) from news text. The main limitation: opposites are close (happy/sad 0.70 vs happy/glad 0.75), because they appear in the same contexts. In all models joy is closest to sadness, and the emotions we confused in project 1 (anger–disgust, shame–guilt) are also close, so embeddings alone will not separate them easily.

## 3. Embed-and-pool
Average the word vectors, then classify. Things to try: frozen vs. trainable embeddings, layer sizes, dropout, learning rate.

## 4. LSTM
Read the words in order. Things to try: hidden size, frozen vs. trainable embeddings, amount of training data. Watch for overfitting.

## 5. Hyperparameter search and variance (together)
How we choose settings, and the Bayesian comparison with a ROPE. Choose settings on a separate dev split, not on the test fold.

## 6. Error analysis (together)
Which kinds of texts does each model get wrong? For example: length, unknown words, negation, emotion words.